# 对应 tf.keras 的 01～02 章节

In [6]:
import matplotlib as mpl
import matplotlib.pyplot as plt
"""
    这是Jupyter Notebook/JupyterLab的魔法命令（magic command），只能在Jupyter里面用，普通python脚本（.py文件）写了会报错
    inline：内联。让matplotlib画出来的图直接嵌入在网页单元格输出里面，而不是弹出独立的窗口
"""
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
"""
    os是Python内置标准库，不需要pip安装，Python自带
    作用：用来和操作系统交互，操作文件、文件夹、路径、环境变量等
"""
import os
"""
    sys是Python内置标准库，不需要pip安装，直接import
    用来和Python解释器本身交互，获取解释器信息、命令行参数、标准输入输出、模块搜索路径等
"""
import sys
"""
    time是Python内置标准库，不用pip安装，直接导入
    作用：处理时间、计时、延时、机器学习代码里经常用来计算程序运行耗时
"""
import time
"""
    tqdm：用来给循环加进度条（机器学习训练、批量处理数据非常常用）
    auto：自动探测当前运行环境
    如果在JupyterLab/Notebook -> 自动加载 tqdm.notebook，显示漂亮的交互式进度条（绿色，不会刷屏）
    如果在终端/PyCharm普通脚本 -> 自动加载普通tqdm，文本进度条
"""
from tqdm.auto import tqdm

"""
    PyTorch的顶层主包 
    提供张量（Tensor）、GPU运算、自动微分、随机种子等最基础功能
"""
import torch
"""
    nn = neural network (神经网络)    
    torch.nn里面是类：网络层、模型容器
    nn.Linear, nn.ReLu, nn.Conv2d, nn.CrossEntropyLoss, nn.Sequential
"""
import torch.nn as nn
"""
    F里面是函数版本的层，激活函数
    函数直接调用，不需要提前实例化
    F.relu, F.sigmoid, F.softmax, F.cross_entropy
    
    nn和F的核心对比
    nn.ReLu()：类，需要创建对象，一般写在模型__init__
    F.relu()：函数，直接调用，写在forward前向传播里
"""
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
# mps = Metal Performance Shaders，苹果硅GPU加速后端
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(device)

sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.11.2
numpy 2.5.3
pandas 3.0.5
sklearn 1.9.1
torch 2.14.0
mps


In [7]:
28 * 28

784

# 数据准备

In [9]:
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision import transforms
"""
    定义数据集的变换
    transforms.Compose：把多个图像操作按顺序打包，依次执行
    transforms.ToTensor():
        输入：PIL图片/numpy数组，像素范围[0, 255]，形状 [H, W, C]
        输出：torch张量，像素缩放至 [0, 1]，自动调换维度为 [C, H, W] （PyTorch卷积要求格式）
        这一步只是缩放到0～1，不是Z-score标准化
    C：Channel 通道
        灰度图（FashionMNIST, MNIST）：C=1，只有亮度信息，没有颜色
        RGB彩色图（CIFAR10）：C=3，分别是红、绿、蓝3个通道
    H：Height 高度
        图片有多少行像素
        FashionMNIST图片：H=28
    W：Width宽度
        图片有多少列像素
        FashionMNIST图片：W=28
    FashionMNIST单张图片张量形状：[1, 28, 28] -> C=1, H=28, W=28
"""
transform = transforms.Compose([ToTensor(),  # 转换为tensor，进行归一化
                                # transforms.Normalize(mean, std)   标准化，mean和std是数据集的均值和方差
                                ])
"""
    fashion_mnist图像分类数据集，衣服分类，60000张训练图片，10000张测试图片
    衣物灰度图片分类数据集，一共10类，图片大小 28 x 28，训练集 60000 张，测试集 10000 张
    root='data':
        数据集存放的根文件夹。运行后会在当前目录创建 data/FashionMNIST，里面放raw原始文件和processed处理好的数据
    train=True:
        True -> 加载训练集（6万张）
        False -> 加载测试集（1万张），写测试集的时候改成 train=False
    download=True:
        第一次运行：自动联网下载数据集到root目录
        已经存在文件时，不会重复下载，安全
        离线跑代码可以改成 download=False
    transform=transform:
        对图片做预处理。输入的是PIL图像，在这里写变换流水线
"""
train_ds = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)
test_ds = datasets.FashionMNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)
"""
    torchvision 数据集里没有提供训练集和验证集的划分
    当然也可以用 torch.utils.data.Dataset 实现人为划分
"""

'\n    torchvision 数据集里没有提供训练集和验证集的划分\n    当然也可以用 torch.utils.data.Dataset 实现人为划分\n'